In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf

print(f"TensorFlow GPU Setup Status: {tf.config.list_physical_devices('GPU')}")

# Load the 100,000 synthetic proteins your CPU generated earlier
df = pd.read_csv("deep_protein_dataset.csv")

print(f"Total Sequences Loaded: {len(df)}")
df.head()


TensorFlow GPU Setup Status: []
Total Sequences Loaded: 100000


,Protein_Sequence,Is_Pathogenic
0,RSGQSPYWNDFHNSYVIKAVENFWKDTNMVYFPTGLYVRSKVQGIN...,1
1,QPEFAYIYHVTIMFIYKHQNVRWQAIMNYKHKDMFHIICNFEQMII...,0
2,AYSDENMIPDWYRGCDEEFIFNCWYKYTWSNYDAMYKYSWSYWMVT...,1
3,HMNFTIHSLAGWTEGCEMVCNQTHIWDDKTPWSRPIQYEVQHMVYI...,1
4,APTPFTQAKIIKCKIYDYASSNMMIILRNWVHMLMDIQMCTTRRPQ...,1


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Build the Translator (Read Character by Character)
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts(df["Protein_Sequence"])

# 2. Translate the Letters into Numbers
numerical_sequences = tokenizer.texts_to_sequences(df["Protein_Sequence"])

# 3. Force all sequences to be exactly 300 digits long 
# (If it's too short, stick zeros at the end to pad it out)
max_protein_length = 300
X = pad_sequences(numerical_sequences, maxlen=max_protein_length, padding='post')

# The Targets (Cancer=1, Healthy=0)
y = df["Is_Pathogenic"].values

print("--- The Translator is Finished ---")
print(f"The Dictionary: {tokenizer.word_index}")
print("\nLook at Sequence #1 before translation:")
print(df["Protein_Sequence"][0])
print("\nLook at Sequence #1 AFTER translation:")
print(X[0])


--- The Translator is Finished ---
The Dictionary: {'g': 1, 'a': 2, 't': 3, 'm': 4, 'e': 5, 's': 6, 'r': 7, 'k': 8, 'f': 9, 'q': 10, 'd': 11, 'i': 12, 'n': 13, 'y': 14, 'v': 15, 'c': 16, 'l': 17, 'w': 18, 'p': 19, 'h': 20}

Look at Sequence #1 before translation:
RSGQSPYWNDFHNSYVIKAVENFWKDTNMVYFPTGLYVRSKVQGINCETGCKSWNRDDDIFEKKEAVPWLWEMDVFWIPSRRDIFPKSTGEWCWRTYLHSIHRIYHKWPWSKEAIIFCLHFLDTVMYWPKQPSHGCEMKTPFMNKKAMWSCQECKKAPAFMRTHDWFFKYYGTCCAPDCNFHAHIDDSTVWMRQYKSYYGTFLNKEDYEHMMIILKIFLWIQFDAHDMSWTQMSPVFEPQNPDNSGHIVQVA

Look at Sequence #1 AFTER translation:
[ 7  6  1 10  6 19 14 18 13 11  9 20 13  6 14 15 12  8  2 15  5 13  9 18
  8 11  3 13  4 15 14  9 19  3  1 17 14 15  7  6  8 15 10  1 12 13 16  5
  3  1 16  8  6 18 13  7 11 11 11 12  9  5  8  8  5  2 15 19 18 17 18  5
  4 11 15  9 18 12 19  6  7  7 11 12  9 19  8  6  3  1  5 18 16 18  7  3
 14 17 20  6 12 20  7 12 14 20  8 18 19 18  6  8  5  2 12 12  9 16 17 20
  9 17 11  3 15  4 14 18 19  8 10 19  6 20  1 16  5  4  8  3 19  9  4 13
  8  

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense

# The size of our alphabet (20 standard amino acids + padding zeros)
vocab_size = len(tokenizer.word_index) + 1 

# 1. Initialize the empty Neural Network
model = Sequential()

# 2. Add the Embedding Layer (Translates the dictionary into a 3D matrix)
model.add(Embedding(input_dim=vocab_size, output_dim=8, input_length=max_protein_length))

# 3. Add the 1D CNN Scanner (Slide a 5-letter window across the sequence)
model.add(Conv1D(filters=32, kernel_size=5, activation='relu'))

# 4. Shrink the massive data down to the most important detected features
model.add(GlobalMaxPooling1D())

# 5. Output Layer: The Final Sigmoid Percentage (Sick or Healthy?)
model.add(Dense(1, activation='sigmoid'))

# Assemble the Calculus Engine
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the architecture blueprint
model.summary()




Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 300, 8)            168       
                                                                 
 conv1d (Conv1D)             (None, 296, 32)           1312      
                                                                 
 global_max_pooling1d (Glob  (None, 32)                0         
 alMaxPooling1D)                                                 
                                                                 
 dense (Dense)               (None, 1)                 33        
                                                                 
Total params: 1513 (5.91 KB)
Trainable params: 1513 (5.91 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [6]:
import time
print("Starting the Deep Learning Engine...")
start_training = time.time()

# 6. Hitting the Gas Pedal (Start the Math)
history = model.fit(
    X,            # The 300-length translated sequences
    y,            # The 1 or 0 labels
    epochs=5,     # Loop over the 100,000 proteins 5 complete times
    batch_size=64,# Push 64 sequences through the math at the exact same time
    validation_split=0.2 # Hide 20% of the proteins so it can't cheat on the final test
)

print(f"\nTraining Complete! Time Taken: {time.time() - start_training:.2f} seconds")


Starting the Deep Learning Engine...
Epoch 1/5


1250/1250 [==============================] - 5s 4ms/step - loss: 0.4341 - accuracy: 0.7693 - val_loss: 0.2798 - val_accuracy: 0.8638
Epoch 2/5
1250/1250 [==============================] - 4s 4ms/step - loss: 0.2693 - accuracy: 0.8685 - val_loss: 0.2629 - val_accuracy: 0.8698
Epoch 3/5
1250/1250 [==============================] - 4s 4ms/step - loss: 0.2562 - accuracy: 0.8721 - val_loss: 0.2641 - val_accuracy: 0.8702
Epoch 4/5
1250/1250 [==============================] - 5s 4ms/step - loss: 0.2463 - accuracy: 0.8748 - val_loss: 0.2441 - val_accuracy: 0.8749
Epoch 5/5
1250/1250 [==============================] - 5s 4ms/step - loss: 0.1791 - accuracy: 0.9279 - val_loss: 0.1465 - val_accuracy: 0.9491

Training Complete! Time Taken: 23.75 seconds
